# Ingresos

In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── 1. Carga datos
df = pd.read_csv('Ingresos Troncal.csv', encoding='latin1')

# Limpiar columnas de monto
for col in ['Monto Total Efectivo', 'Monto Total Tarjetas']:
    df[col] = df[col].replace(r'[\$,]', '', regex=True)
    df[col] = df[col].replace(r'^\s*-\s*$', '0', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['Ingreso Total'] = df['Monto Total Efectivo'] + df['Monto Total Tarjetas']
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Serie diaria
ts_diaria = df.groupby('Fecha')['Ingreso Total'].sum()
ts_diaria = ts_diaria.asfreq('D').interpolate()

# Filtro 2023–2025 de tiempo
ts_dia = ts_diaria[(ts_diaria.index >= '2023-01-01') &
                   (ts_diaria.index <= '2025-12-31')].copy().to_frame(name='Ingreso')

# ── 2. Promedio e índice estacional por día de semana
DIAS = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

ts_dia['DiaSemana'] = ts_dia.index.dayofweek   # 0=Lun … 6=Dom
ts_dia['DiaNombre'] = ts_dia['DiaSemana'].map(dict(enumerate(DIAS)))

promedio_dia  = ts_dia.groupby('DiaSemana')['Ingreso'].mean()
media_global  = ts_dia['Ingreso'].mean()
indice_dia    = (promedio_dia / media_global * 100).rename('Índice')

mejor_dia = indice_dia.idxmax()


for d, idx in zip(DIAS, indice_dia):
    barra = '█' * int(idx / 5)
    marca = '  ◄ MÁXIMO' if idx == indice_dia.max() else ''
    print(f"  {d:<10}  {idx:6.1f}  {barra}{marca}")

# ── 3. Gráfica: barras + boxplot ─────────────────────────────────────────────
BLUE  = '#1C4E80'
GREEN = '#21A659'
RED   = '#E74C3C'
BG    = '#F4F7FB'

colores = [GREEN if i == indice_dia.idxmax() else
           RED   if i == indice_dia.idxmin() else
           BLUE  for i in indice_dia.index]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=BG)
fig.suptitle('Estacionalidad Semanal — Ingreso Total Troncal 2023–2025',
             fontsize=14, fontweight='bold', color=BLUE, y=1.01)

# ── Panel izquierdo: Índice estacional 
ax = axes[0]
ax.set_facecolor('white')
bars = ax.bar(DIAS, indice_dia.values, color=colores,
              edgecolor='white', linewidth=0.8, width=0.62, zorder=3)
ax.axhline(100, color='black', linestyle='--', lw=1.2, label='Promedio = 100')
ax.set_title('Índice estacional por día de la semana',
             fontweight='bold', fontsize=11, color=BLUE, pad=8)
ax.set_ylabel('Indice Estacional (base 100)', fontsize=10)
ax.set_ylim(0, indice_dia.max() * 1.18)
ax.legend(fontsize=9, framealpha=0.7)
ax.grid(axis='y', alpha=0.35, zorder=0)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='x', labelsize=9)
for i, v in enumerate(indice_dia.values):
    ax.text(i, v + 1.2, f'{v:.1f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#1a1a1a')
ax.text(0.5, -0.15,
        f'🏆  Día con más ingresos: {DIAS[mejor_dia]}',
        transform=ax.transAxes, ha='center',
        fontsize=11, fontweight='bold', color=GREEN)

# ── Panel derecho: Boxplot ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('white')
data_box = [ts_dia[ts_dia['DiaSemana'] == d]['Ingreso'].values / 1e6
            for d in range(7)]
bp = ax2.boxplot(data_box, labels=DIAS, patch_artist=True,
                 medianprops=dict(color='white', lw=2.2))
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.82)
ax2.set_title('Distribución de ingresos por día (millones MXN)',
              fontweight='bold', fontsize=11, color=BLUE, pad=8)
ax2.set_ylabel('Millones MXN', fontsize=10)
ax2.grid(axis='y', alpha=0.35)
ax2.spines[['top', 'right']].set_visible(False)
ax2.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('grafica_dow_ingreso.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
print("\n✅  Guardada: grafica_dow_ingreso.png")

  Lunes        117.8  ███████████████████████  ◄ MÁXIMO
  Martes       111.7  ██████████████████████
  Miércoles    106.3  █████████████████████
  Jueves       105.2  █████████████████████
  Viernes      108.9  █████████████████████
  Sábado        92.3  ██████████████████
  Domingo       57.7  ███████████

✅  Guardada: grafica_dow_ingreso.png


# PASAJEROS



In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

# ── 1. Carga y preparación ────────────────────────────────────────────────────
df = pd.read_csv('Ingresos Troncal.csv', encoding='latin1')
df['Total Pasajes'] = df['Total Pasajes'].str.replace(',', '').astype(float)
df['Fecha'] = pd.to_datetime(df['Fecha'])

daily = df.groupby('Fecha')['Total Pasajes'].sum()
daily.index = pd.DatetimeIndex(daily.index, freq='D')

# ── 2. Entrenamiento 2023–2025 ────────────────────────────────────────────────
train = daily['2023':'2025']

# ── 3. Modelo SARIMA(0,1,1)(0,1,1)[7] ────────────────────────────────────────
model  = SARIMAX(train,
                 order=(0, 1, 1),
                 seasonal_order=(0, 1, 1, 7),
                 enforce_stationarity=False,
                 enforce_invertibility=False)
result = model.fit(disp=False)
print(result.summary())

# ── 4. Promedio por día de semana (SIN excluir festivos) ──────────────────────
DOW_ES  = {0: 'Lun', 1: 'Mar', 2: 'Mié', 3: 'Jue', 4: 'Vie', 5: 'Sáb', 6: 'Dom'}
DOW_ORD = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

t = train.copy().to_frame()
t['dow'] = t.index.dayofweek

dow_avg = (t.groupby('dow')['Total Pasajes']
            .mean()
            .rename(index=DOW_ES)
            .reindex(DOW_ORD))

best_day = dow_avg.idxmax()
print("\n── Promedio pasajes por día de semana (todos los días) ──")
for day, val in dow_avg.items():
    flag = "  ◄ MÁXIMO" if day == best_day else ""
    print(f"  {day}: {val:,.0f}{flag}")

# ── 5. Gráfica de barras ──────────────────────────────────────────────────────
BLUE  = '#1C4E80'
GREEN = '#21A659'
BG    = '#F4F7FB'

fig, ax = plt.subplots(figsize=(9, 6), facecolor=BG)
t = train.copy().to_frame()
t['dow'] = t.index.dayofweek

dow_avg = (t.groupby('dow')['Total Pasajes']
            .mean()
            .rename(index=DOW_ES)
            .reindex(DOW_ORD))



colors = [GREEN if d == best_day else BLUE for d in dow_avg.index]

bars = ax.bar(dow_avg.index, dow_avg.values / 1000,
              color=colors, edgecolor='white', linewidth=0.8,
              width=0.62, zorder=3)

# Etiquetas sobre cada barra
for bar, val in zip(bars, dow_avg.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val / 1000:.1f}k',
            ha='center', va='bottom',
            fontsize=9.5, fontweight='bold', color='#1a1a1a')

# Títulos y ejes
ax.set_title('Promedio de pasajes por día de semana\n(2023–2025)',
             fontsize=13, fontweight='bold', color=BLUE, pad=12)
ax.set_ylabel('Pasajes (miles)', fontsize=10, color='#333')
ax.set_ylim(0, dow_avg.max() / 1000 * 1.20)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.yaxis.set_tick_params(left=False)

# Nota al pie
ax.text(0.5, -0.12,
        f'🏆  Día con más ingresos:  {best_day}',
        transform=ax.transAxes, ha='center',
        fontsize=12, fontweight='bold', color=GREEN)

# Nota modelo
ax.text(0.99, 0.97,
        'Modelo: SARIMA(0,1,1)(0,1,1)[7]',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=8, color='#777',
        bbox=dict(boxstyle='round,pad=0.3', fc='#F0F4F8', ec='#ccc', lw=0.8))

plt.tight_layout()
plt.savefig('grafica_dow_sarima.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
print("\n✅  Guardada como 'grafica_dow_sarima.png'")


                                     SARIMAX Results                                     
Dep. Variable:                     Total Pasajes   No. Observations:                 1096
Model:             SARIMAX(0, 1, 1)x(0, 1, 1, 7)   Log Likelihood              -11723.072
Date:                           Sat, 21 Mar 2026   AIC                          23452.145
Time:                                   14:41:15   BIC                          23467.096
Sample:                               01-01-2023   HQIC                         23457.806
                                    - 12-31-2025                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.6696      0.022    -30.864      0.000      -0.712      -0.627
ma.S.L7       -0.7625      0.018    -42.864